# Ejecutar modelos guardados CatBoost

Notebook reducido para cargar datos, ejecutar modelos CatBoost guardados y exportar resultados.


In [2]:
#Cargar librerias generales y documento para el historial


import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

#-- Activar estas importaciones si quiero generar el docx con las capturas ---
#import importlib, util_docx
#from util_docx import captura_a_docx
#importlib.reload(util_docx) 

#prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx"
#historial = "detalle matricula cohorte 2019.xlsx"
historial = "historia_todos.csv" #Este archvio es del 2019 al 2025-10
hitorial_parqet= "historia_todos.parquet"
#historial = "historia_todos_2014_202510_2025-10-22202332.csv"
                                
from IPython.display import display


In [4]:
## check asgianturas en carpeta

import os

def check_asignaturas_en_carpeta(ruta_carpeta_procesados):
    """
    Revisa el contenido de una carpeta y devuelve una lista con los nombres
    de los archivos sin extensión.

    Parámetros:
    ----------
    ruta_carpeta_procesados : str
        Ruta al directorio que se desea revisar.

    Retorna:
    -------
    list
        Lista con los nombres de los archivos sin extensión. 
        Si la carpeta no existe, devuelve una lista vacía y muestra un mensaje.
    """
    if not os.path.exists(ruta_carpeta_procesados):
        print(f"⚠️ La carpeta '{ruta_carpeta_procesados}' no existe.")
        return []

    archivos = [
        os.path.splitext(nombre)[0]
        for nombre in os.listdir(ruta_carpeta_procesados)
        if os.path.isfile(os.path.join(ruta_carpeta_procesados, nombre))
    ]
    
    archivos_df=pd.DataFrame(archivos,columns=['asignatura'])
    print(f"Lista de las asignaturas en la carpeta ")
    display(archivos_df)

    return archivos

In [5]:
## elegir asignaturas

def elegir_asignaturas(historial,cat_prereq , cant_asignaturas=5):
    
    df_historial = historial[ (historial["Observacion_Prerrequisito"] == cat_prereq)]

    df_filtrado = (
        df_historial
        .groupby("Cod materia curso")
        .size()
        .reset_index(name="num_registros")
        .sort_values("num_registros", ascending=False)
        .reset_index(drop=True)
    )

    lista_asignaturas=df_filtrado["Cod materia curso"].head(cant_asignaturas).tolist()    

    return lista_asignaturas

def elegir_asignaturas_rev_ya_creados(historial, cat_prereq, ruta_modelos, cant_asignaturas=5):

    if cat_prereq=="all":
        df_historial= historial[(historial["Observacion_Prerrequisito"] == "Prerrequisito cumplido") | (historial["Observacion_Prerrequisito"] == "No tiene pre requisito")]
    else:
        df_historial = historial[ (historial["Observacion_Prerrequisito"] == cat_prereq)]

    lista_mod_creados=check_asignaturas_en_carpeta(ruta_modelos)

    df_filtrado = (
        df_historial
        .groupby("Cod materia curso")
        .size()
        .reset_index(name="num_registros")
        .sort_values("num_registros", ascending=False)
        .reset_index(drop=True)
    )

    #print(lista_mod_creados)
    #lista_asignaturas=df_filtrado[df_filtrado["Cod materia curso"].isin(lista_mod_creados)].head(cant_asignaturas).tolist()    
    #lista_asignaturas=df_filtrado[df_filtrado["Cod materia curso"].isin(lista_mod_creados)]
    lista_asignaturas= df_filtrado[~(df_filtrado["Cod materia curso"].isin(lista_mod_creados))].head(cant_asignaturas)
    print(f"Lista de asignaturas seleccionadas: \n {lista_asignaturas}")
    lista_asignaturas = lista_asignaturas['Cod materia curso'].tolist() 

    return lista_asignaturas

In [6]:
## Crear DataFrame con los nombres de las asignaturas y sus códigos

def crear_df_asignaturas(df_historial: pd.DataFrame) -> pd.DataFrame:
    # Extraer las columnas relevantes
    df_asignaturas = df_historial[['Cod materia curso', 'Descripcion_Materia']].drop_duplicates()
    
    # Eliminar filas con valores nulos en 'Cod materia curso' o 'Descripcion_Materia'
    df_asignaturas = df_asignaturas.dropna(subset=['Cod materia curso', 'Descripcion_Materia'])
    
    # Eliminar espacios en blanco al inicio y al final de los nombres de las asignaturas
    df_asignaturas['Descripcion_Materia'] = df_asignaturas['Descripcion_Materia'].str.strip()
    
    # Eliminar duplicados basados en 'Cod materia curso', manteniendo la primera ocurrencia
    df_asignaturas = df_asignaturas.drop_duplicates(subset=['Cod materia curso'], keep='first')
    
    # Ordenar el DataFrame por 'Cod materia curso'
    df_asignaturas = df_asignaturas.sort_values(by='Cod materia curso').reset_index(drop=True)

    print(f"👌 Creado DataFrame con las asignaturas unicas y sus nombres. Número de asignaturas únicas: {len(df_asignaturas)}")
    
    return df_asignaturas

In [7]:
## Calcular número de intentos por asignatura

def calcular_num_intentos(df: pd.DataFrame) -> pd.DataFrame:
    # Paso 1: quedarse con intentos únicos por periodo
    df_unique = df[['Pidm', 'Cod materia curso', 'Periodo']].drop_duplicates()

    # Paso 2: ordenar por periodo
    df_unique = df_unique.sort_values(['Pidm', 'Cod materia curso', 'Periodo'])

    # Paso 3: generar número de intento acumulado
    # rank(method='dense') da consecutivos (1,2,3..) por periodo distinto
    df_unique['num_intentos_asignatura'] = (
        df_unique.groupby(['Pidm', 'Cod materia curso']).cumcount() + 1
    )

    # Paso 4: hacer merge con el dataframe original
    df_result = df.merge(
        df_unique,
        on=['Pidm', 'Cod materia curso', 'Periodo'],
        how='left'
    )

    # --- Nuevo segmento: número de semestres que un profesor dictó la asignatura ---
    # Buscar el nombre de la columna de profesor disponible (compatibilidad)
    posibles_cols_prof = ['Prof_Codigo', '_ Matricula detalle para analisis.Prof_Codigo', 'profesor_codigo']
    profesor_col = next((c for c in posibles_cols_prof if c in df_result.columns), None)

    if profesor_col is not None:
        # Paso A: quedarse con combinaciones únicas profesor - asignatura - periodo
        df_prof_unique = df_result[[profesor_col, 'Cod materia curso', 'Periodo']].drop_duplicates()

        # Paso B: ordenar por profesor, asignatura y periodo
        df_prof_unique = df_prof_unique.sort_values([profesor_col, 'Cod materia curso', 'Periodo'])

        # Paso C: contar semestres únicos acumulados para cada profesor-asignatura
        df_prof_unique['num_semestres_profesor_asignatura'] = (
            df_prof_unique.groupby([profesor_col, 'Cod materia curso']).cumcount() + 1
        )

        # Paso D: merge con el dataframe resultante
        df_result = df_result.merge(
            df_prof_unique,
            on=[profesor_col, 'Cod materia curso', 'Periodo'],
            how='left'
        )
    else:
        # Si no existe columna de profesor, crear la columna con NaN y avisar (silencioso)
        df_result['num_semestres_profesor_asignatura'] = pd.NA

    return df_result

In [8]:
## Funciones de limpieza del DataFrame

def arreglar_comas_por_puntos(df: pd.DataFrame, cols_excluir: list) -> pd.DataFrame:
    """
    Reemplaza comas (,) por puntos (.) en todas las columnas tipo string,
    excepto en las columnas listadas en cols_excluir.
    Intenta convertir los valores resultantes a float.
    """
    for col in df.columns:
        if col not in cols_excluir and df[col].dtype == object:
            df[col] = df[col].str.replace(",", ".", regex=False)
            # Intentar conversión a float cuando sea posible
            try:
                df[col] = df[col].astype(float)
            except ValueError:
                pass  # si no se puede convertir, se queda como string
            print(f"[Comas→Puntos] Procesada columna: {col}")
        elif col in cols_excluir:
            print(f"[Comas→Puntos] Columna excluida: {col}")
    return df


def imputar_valores(df: pd.DataFrame) -> pd.DataFrame:
    """Imputa valores en columnas específicas."""
    # Columna repitencia → vacíos a 0.0
    col_repitencia = "_ Matricula detalle para analisis.repitencia profesor referencia"
    if col_repitencia in df.columns:
        antes = df[col_repitencia].isna().sum()
        df[col_repitencia] = df[col_repitencia].fillna(0).astype(float)
        despues = df[col_repitencia].isna().sum()
        print(f"[Imputación] Columna '{col_repitencia}': {antes} vacíos reemplazados por 0.0")

        
    # Columna repitencia profesor 1 año → vacíos a 0.0
    col_repitencia = "_ Matricula detalle para analisis.repitencia profesor referencia 1ano"
    if col_repitencia in df.columns:
        antes = df[col_repitencia].isna().sum()
        df[col_repitencia] = df[col_repitencia].fillna(0).astype(float)
        despues = df[col_repitencia].isna().sum()
        print(f"[Imputación] Columna '{col_repitencia}': {antes} vacíos reemplazados por 0.0")

    # Columna Procedencia Categoria → reemplazar "6 No registra" por NaN
    col_procedencia = "_ Matricula detalle para analisis.Procedencia Categoria"
    if col_procedencia in df.columns:
        conteo = (df[col_procedencia] == "6 No registra").sum()
        df[col_procedencia] = df[col_procedencia].replace("6 No registra", np.nan)
        print(f"[Imputación] Columna '{col_procedencia}': {conteo} valores '6 No registra' reemplazados por NaN")

    # Columna Sexo → reemplazar -99 por NaN
    col_sexo = "_ Matricula detalle para analisis.Sexo"
    if col_sexo in df.columns:
        conteo = (df[col_sexo] == -99).sum()
        df[col_sexo] = df[col_sexo].replace(-99, np.nan)
        print(f"[Imputación] Columna '{col_sexo}': {conteo} valores '-99' reemplazados por NaN")

    # Columna Calificacion_Final → vacíos a -1
    col_nota_final = "Calificacion_Final"
    if col_nota_final in df.columns:
        conteo = df[col_nota_final].isna().sum()
        df[col_nota_final] = df[col_nota_final].fillna(-1).astype(float)
        df[col_nota_final] = df[col_nota_final].apply(lambda x : -1 if x <0 else x )
        print(f"[Imputación] Columna '{col_nota_final}': {conteo} vacíos reemplazados por -1")

    print("[Creacion] de columna categorica para retiros")
    col_nota_final_rubiel="_ Matricula detalle para analisis.Calif Final _ Retiros"
    col_retiro_cat="Retiro_Asignatura_Cat"
    #df[col_retiro_cat]= df[col_nota_final].apply(lambda x: 1 if x <0 else 0)
    df[col_retiro_cat]= df[col_nota_final_rubiel].apply(lambda x: 1 if x <0 else 0)

    # Añadir columna de numero de intentos asignatura
    df = calcular_num_intentos(df)
    print(f"[Imputación] Columna num_intentos_asignatura': usando funcion calcular_num_intentos")
    print(f"[Imputación] Columna num_semestres_profesor_asignatura': usando funcion calcular_num_intentos")

    return df


def eliminar_columnas_vacias(df: pd.DataFrame, columnas_eliminadas: list) -> pd.DataFrame:
    """Elimina columnas 100% vacías."""
    cols_vacias = df.columns[df.isna().all()].tolist()
    if cols_vacias:
        df = df.drop(columns=cols_vacias)
        columnas_eliminadas.extend(cols_vacias)
        print(f"[Columnas] Eliminadas columnas vacías: {cols_vacias}")
    else:
        print("[Columnas] No se encontraron columnas totalmente vacías")
    return df


def eliminar_filas_por_columna(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Elimina filas con nulos en columnas con <20% vacíos."""
    filas_eliminadas_total = 0
    total_filas_inicial = len(df)

    for col in df.columns:
        nulos = df[col].isna().sum()
        porcentaje = nulos / len(df) if len(df) > 0 else 0

        if porcentaje == 0:
            continue

        if porcentaje < 0.20:
            filas_antes = len(df)
            df = df.dropna(subset=[col])
            filas_despues = len(df)
            eliminadas = filas_antes - filas_despues
            filas_eliminadas_total += eliminadas
            print(f"[Filas] Columna '{col}': {porcentaje:.2%} nulos → eliminadas {eliminadas} filas")
        else:
            print(f"[Aviso] Columna '{col}' tiene {porcentaje:.2%} de valores nulos (≥20%), no se eliminaron filas")

    filas_finales = len(df)
    print(f"[Filas] Total eliminadas: {filas_eliminadas_total} (de {total_filas_inicial} → {filas_finales})")

    return df, filas_eliminadas_total

def eliminar_filas_por_columna_ignorando_col_prereq(df: pd.DataFrame, col_add_ignorar=False) -> tuple[pd.DataFrame, int]:
    """
    Elimina filas con nulos en columnas con <20% vacíos,
    ignorando las columnas que siguen el patrón Prereq_i_[Atributo].
    """
    filas_eliminadas_total = 0
    total_filas_inicial = len(df)
    cols_to_ignore = [
        "_ Matricula detalle para analisis.Puntaje_icfes_recalificado",
        "_ Matricula detalle para analisis.Dif anios icfes clase",
        "_ Matricula detalle para analisis.PCN_Puntaje_Ciencias_Naturales",
        "_ Matricula detalle para analisis.PIN_Puntaje_en_Ingles",
        "_ Matricula detalle para analisis.PLC_Puntaje_para_Lectura_Critica",
        "_ Matricula detalle para analisis.PMA_Puntaje_para_Matematicas",
        "_ Matricula detalle para analisis.PSC_Puntaje_Sociales_y_Ciudadanas",
        "_ Matricula detalle para analisis.Tipo_Colegio",
        "_ Matricula detalle para analisis.Tipo_Calendario",
    ]

    for col in df.columns:
        # Ignorar columnas de prerequisitos
        if col.startswith("Prereq_"):
            print(f"[Ignorado] Columna '{col}' pertenece a prerequisitos, no se revisa.")
            continue

        if col in cols_to_ignore and col_add_ignorar:
            print(f"[Ignorado] Columna '{col}' pertenece a columnas a ignorar, no se revisa.")
            continue

        nulos = df[col].isna().sum()
        porcentaje = nulos / len(df) if len(df) > 0 else 0

        if porcentaje == 0:
            continue

        if porcentaje < 0.20:
            filas_antes = len(df)
            df = df.dropna(subset=[col])
            filas_despues = len(df)
            eliminadas = filas_antes - filas_despues
            filas_eliminadas_total += eliminadas
            print(f"[Filas] Columna '{col}': {porcentaje:.2%} nulos → eliminadas {eliminadas} filas")
        else:
            print(f"[Aviso] Columna '{col}' tiene {porcentaje:.2%} de valores nulos (≥20%), no se eliminaron filas")

    filas_finales = len(df)
    print(f"[Filas] Total eliminadas: {filas_eliminadas_total} (de {total_filas_inicial} → {filas_finales})")

    return df, filas_eliminadas_total



def limpiar_dataframe(df: pd.DataFrame, ignorar_col_adicionales = False) -> pd.DataFrame:
    """Función principal que limpia el dataframe según reglas definidas."""
    print("=== Iniciando limpieza del DataFrame ===")

    # Imputación de valores
    df = imputar_valores(df)

    # Eliminar columnas vacías
    columnas_eliminadas = []
    df = eliminar_columnas_vacias(df, columnas_eliminadas)

    # Eliminar filas según condición por columnas
    #df, filas_eliminadas = eliminar_filas_por_columna(df)
    df, filas_eliminadas = eliminar_filas_por_columna_ignorando_col_prereq(df,ignorar_col_adicionales)
    
    # Resumen final
    print("\n=== Resumen limpieza ===")
    print(f"Filas eliminadas: {filas_eliminadas}")
    print(f"Columnas eliminadas: {columnas_eliminadas}")

    return df


In [9]:
#Renombrar Columnas

def renombrar_columnas(df: pd.DataFrame, tiene_prereq=True):
    """
    Renombra columnas largas por versiones cortas en snake_case.
    Devuelve:
      - DataFrame con columnas renombradas
      - Lista con columnas finales de análisis (col_usar renombradas)
      - String con el nombre de la variable objetivo renombrada
    """

    # Mapeo de nombres originales a nuevos
    mapping = {
        "_ Matricula detalle para analisis.repitencia profesor referencia 1ano": "repitencia_prof_ref",
        "Nombre_Programa": "programa",
        "_ Matricula detalle para analisis.Prof_Codigo": "profesor_codigo",
        "_ Matricula detalle para analisis.pga inicial": "pga_inicial",
        "_ Matricula detalle para analisis.prom semestral t_1": "promedio_sem_t1",
        "_ Matricula detalle para analisis.Sexo": "sexo",
        "_ Matricula detalle para analisis.Asistencia CREE t_1": "asistencia_cree_t1",
        "_ Matricula detalle para analisis.Procedencia Categoria": "procedencia_categoria",
        "_ Matricula detalle para analisis.Edad cursan asignatura": "edad_curso",
        "_ Matricula detalle para analisis.Calif Final _ Retiros": "resultado_final",
        "_ Matricula detalle para analisis.Puntaje estrato" : "estrato",
        "_ Matricula detalle para analisis.Puntaje_icfes_recalificado" : "puntaje_saber11",
        "_ Matricula detalle para analisis.Dif anios icfes clase" : "años_saber11_vs_clase",
        "_ Matricula detalle para analisis.PCN_Puntaje_Ciencias_Naturales" : "saber11_Ciencias_Naturales",
        "_ Matricula detalle para analisis.PIN_Puntaje_en_Ingles": "saber11_ingles",
        "_ Matricula detalle para analisis.PLC_Puntaje_para_Lectura_Critica": "saber11_lectura_critica",
        "_ Matricula detalle para analisis.PMA_Puntaje_para_Matematicas": "saber11_matematicas",
        "_ Matricula detalle para analisis.PSC_Puntaje_Sociales_y_Ciudadanas": "saber11_sociales",
        "_ Matricula detalle para analisis.Tipo_Colegio": "Tipo_colegio",
        "_ Matricula detalle para analisis.Tipo_Calendario": "Tipo_calendario"
    }

      # Renombrar columnas
    df = df.rename(columns=mapping)

    if tiene_prereq:
    # Construir lista final de columnas de análisis
        col_usar = [
            "repitencia_prof_ref",
            "programa",
            "profesor_codigo",
            "pga_inicial",
            "promedio_sem_t1",
            "sexo",
            "asistencia_cree_t1",
            "procedencia_categoria",
            "edad_curso",
            "num_intentos_asignatura",
            "num_semestres_profesor_asignatura",
            "estrato"
            #,"Tipo_colegio","Tipo_calendario"
        ]
    else:
        df,col_usar=check_columnas_no_prereq_pga(df)


    # Variable objetivo
    var_objetivo = "resultado_final"

    return df, col_usar, var_objetivo

def check_columnas_no_prereq_pga(df: pd.DataFrame):
    """
    Verifica si el DataFrame contiene columnas de prerequisitos.
    Retorna True si existen columnas que comienzan con 'Prereq_', False en caso contrario.
    """
    cols_usar=["repitencia_prof_ref",
            "programa",
            "profesor_codigo",
            "sexo",
            "asistencia_cree_t1",
            "procedencia_categoria",
            "edad_curso",
            "num_intentos_asignatura",
            "num_semestres_profesor_asignatura",
            "estrato", "Tipo_colegio", "Tipo_calendario"]
    
    cols_pga_solo=["pga_inicial"]
    
    cols_pga=["pga_inicial", "promedio_sem_t1"]

    cols_icfes_total = ["puntaje_saber11",
        "años_saber11_vs_clase",
        "saber11_Ciencias_Naturales",
        "saber11_ingles",
        "saber11_lectura_critica",
        "saber11_matematicas",
        "saber11_sociales",
    ]

    cols_icfes_sin_pj_general = [ "años_saber11_vs_clase",
        "saber11_Ciencias_Naturales",
        "saber11_ingles",
        "saber11_lectura_critica",
        "saber11_matematicas",
        "saber11_sociales"
    ]
    col_puntje_icfes ="puntaje_saber11"
    
    # Verificar existencia y completitud de la columna pga_inicial
    if "pga_inicial" not in df.columns:
        print("No se encontró la columna 'pga_inicial'")
        return df, cols_usar

    total = len(df)
    if total == 0:
        print("DataFrame vacío")
        return df, cols_usar

    nulos = df["pga_inicial"].isna().sum()
    porcentaje_nulos = nulos / total

    # Evaluar según umbrales
    if porcentaje_nulos > 0.80:
        cols_usar= cols_usar+cols_icfes_total
        print("[❗] Mas del 80% de los datos no tiene PGA usando las variables del icfes y colegio")
    elif abs(porcentaje_nulos > 0.30) :  
        print("[❗] Usar imputacion de ICFES imputar")
        # Intentar imputar pga_inicial usando '_ Matricula detalle para analisis.Puntaje_icfes_recalificado'
        if col_puntje_icfes not in df.columns:
            print(f"[Imputación] No se encuentra la columna '{col_puntje_icfes}', se usarán variables del ICFES por defecto")
            cols_usar = cols_usar + cols_icfes_total
        else:
            # convertir a numérico y trabajar sólo con los valores válidos para min-max
            icfes_ser = pd.to_numeric(df[col_puntje_icfes], errors='coerce')
            icfes_valid = icfes_ser.dropna()

            if icfes_valid.empty or icfes_valid.max() == icfes_valid.min():
                print(f"[Imputación] Imposible normalizar '{col_puntje_icfes}' (valores insuficientes o constantes). Se usan variables ICFES")
                cols_usar = cols_usar + cols_icfes_total
            else:
                min_v = icfes_valid.min()
                max_v = icfes_valid.max()
                # normalizar a [0,1] y escalar a [0,5]
                icfes_scaled = (icfes_ser - min_v) / (max_v - min_v) * 5

                # rellenar sólo donde pga_inicial es nulo
                n_nulos_before = df["pga_inicial"].isna().sum()
                mask_fill = df["pga_inicial"].isna() & icfes_scaled.notna()
                df.loc[mask_fill, "pga_inicial"] = icfes_scaled.loc[mask_fill].astype(float)
                df["pga_inicial"] = df["pga_inicial"].astype(float)

                n_filled = mask_fill.sum()
                print(f"[Imputación] Columna 'pga_inicial': {n_filled} valores imputados usando '{col_puntje_icfes}' (min={min_v}, max={max_v})")

                # Imputar 'promedio_sem_t1' de la misma forma usando icfes_scaled (si existe)
                if "promedio_sem_t1" in df.columns:
                    mask_fill_prom = df["promedio_sem_t1"].isna() & icfes_scaled.notna()
                    df.loc[mask_fill_prom, "promedio_sem_t1"] = icfes_scaled.loc[mask_fill_prom].astype(float)
                    # Asegurar tipo float
                    df["promedio_sem_t1"] = df["promedio_sem_t1"].astype(float)
                    n_filled_prom = mask_fill_prom.sum()
                    print(f"[Imputación] Columna 'promedio_sem_t1': {n_filled_prom} valores imputados usando '{col_puntje_icfes}' (mismo escalado)")
                else:
                    print("[Imputación] Columna 'promedio_sem_t1' no encontrada, no se imputó")

                # ahora podemos usar las columnas pga
                cols_usar = cols_usar + cols_pga_solo + cols_icfes_sin_pj_general
    else:
        cols_usar = cols_usar + cols_pga
        print("[❗] Usar PGA")
      

    return df, cols_usar


In [10]:
## Indentificar columnas de prerequisitos válidas

def columnas_prereq_validas(df: pd.DataFrame, umbral: float = 0.8) -> list:
    """
    Identifica las columnas de prerequisitos (Nota e Intentos) que cumplen con tener
    al menos un 80% de valores no nulos en la columna Nota.
    
    Parámetros:
        df: DataFrame de entrada
        umbral: proporción mínima de valores no nulos (default=0.8)
    
    Retorna:
        lista con las columnas Prereq_i_Nota y Prereq_i_Intentos que cumplen el criterio.
    """
    columnas_seleccionadas = []
    prereq_notas = [c for c in df.columns if c.startswith("Prereq_") and c.endswith("_Nota")]

    for col_nota in prereq_notas:
        col_intentos = col_nota.replace("_Nota", "_Intentos")

        if col_intentos not in df.columns:
            print(f"[Aviso] No se encontró la columna de intentos para {col_nota}, se omite el par.")
            continue

        total = len(df)
        no_nulos = df[col_nota].notna().sum()
        proporcion = no_nulos / total if total > 0 else 0

        if proporcion >= umbral:
            columnas_seleccionadas.extend([col_nota, col_intentos])
            print(f"[Incluido] {col_nota} y {col_intentos} → {proporcion:.2%} valores válidos")
        else:
            print(f"[Excluido] {col_nota} y {col_intentos} → {proporcion:.2%} valores válidos (<{umbral:.0%})")

    print("\nResumen final:")
    print(f"Columnas seleccionadas: {columnas_seleccionadas}")

    return columnas_seleccionadas

## Identificar columnas de prerequisitos válidas (versión 2) Con cambio de nombre!

import unicodedata


def normalizar_nombre(nombre: str) -> str:
    """
    Convierte un nombre en snake_case sin acentos, dejando los números intactos.
    """
    # Eliminar acentos
    nombre = ''.join(
        c for c in unicodedata.normalize('NFD', nombre)
        if unicodedata.category(c) != 'Mn'
    )
    # Reemplazar espacios y caracteres raros por "_", pero dejar letras y números
    nombre = re.sub(r'[^a-zA-Z0-9]+', '_', nombre)
    # Pasar a minúsculas
    nombre = nombre.strip("_").lower()
    return nombre

def columnas_prereq_validas_ext(df: pd.DataFrame, nombres_asignaturas: pd.DataFrame, umbral: float = 0.8):
    """
    Identifica columnas de prerequisitos que cumplen con el umbral de no-nulos
    y crea nuevas columnas con el nombre de la asignatura.

    Parámetros:
        df: DataFrame de entrada (con columnas Prereq_i_Nota y Prereq_i_Codigo)
        nombres_asignaturas: DataFrame con columnas ["Cod materia curso", "Descripcion_Materia"]
        umbral: proporción mínima de valores no nulos en la columna Nota (default=0.8)

    Retorna:
        columnas_nuevas (list): Lista con los nombres de las nuevas columnas creadas
        df_modificado (pd.DataFrame): DataFrame con las columnas nuevas añadidas
    """
    columnas_nuevas = []
    df_modificado = df.copy()

    prereq_notas = [c for c in df.columns if c.startswith("Prereq_") and c.endswith("_Nota")]

    for col_nota in prereq_notas:
        col_intentos = col_nota.replace("_Nota", "_Intentos")
        col_codigo = col_nota.replace("_Nota", "_Codigo")

        if col_intentos not in df.columns:
            print(f"[Aviso] No se encontró la columna de intentos para {col_nota}, se omite el par.")
            continue
        if col_codigo not in df.columns:
            print(f"[Aviso] No se encontró la columna de código para {col_nota}, se omite el par.")
            continue

        total = len(df)
        no_nulos = df[col_nota].notna().sum()
        proporcion = no_nulos / total if total > 0 else 0

        if proporcion >= umbral:
            # Procesar cada fila: buscar nombre de la materia
            nuevas_columnas_temp = []
            for idx, codigo in df.loc[df[col_codigo].notna(), col_codigo].items():
                # Buscar en nombres_asignaturas
                nombre_match = nombres_asignaturas.loc[
                    nombres_asignaturas["Cod materia curso"] == codigo, 
                    "Descripcion_Materia"
                ]
                if not nombre_match.empty:
                    nombre_materia = nombre_match.values[0]
                else:
                    nombre_materia = codigo  # fallback

                # Normalizar nombre
                nombre_materia_norm = normalizar_nombre(str(nombre_materia))
                nueva_columna = f"Prereq_{nombre_materia_norm}_Nota"

                # Crear la columna si no existe
                if nueva_columna not in df_modificado.columns:
                    df_modificado[nueva_columna] = pd.NA

                # Asignar el valor de la nota, asegurando tipo float
                valor_nota = df.loc[idx, col_nota]
                try:
                    valor_nota_float = float(valor_nota)
                except (ValueError, TypeError):
                    valor_nota_float = pd.NA
                df_modificado.at[idx, nueva_columna] = valor_nota_float

                # Forzar tipo float en la columna creada (puede tener <NA>)
                df_modificado[nueva_columna] = pd.to_numeric(df_modificado[nueva_columna], errors='coerce')

                nuevas_columnas_temp.append(nueva_columna)

                # Añadir columna de numero de intentos asociados
                nueva_columna_intentos = f"Prereq_{nombre_materia_norm}_Intentos"
                if nueva_columna_intentos not in df_modificado.columns:
                    df_modificado[nueva_columna_intentos] = pd.NA

                valor_intentos = df.loc[idx, col_intentos]
                try:
                    valor_intentos_float = float(valor_intentos)
                except (ValueError, TypeError):
                    valor_intentos_float = pd.NA
                df_modificado.at[idx, nueva_columna_intentos] = valor_intentos_float

                # Forzar tipo int en la columna de intentos (puede tener <NA>)
                df_modificado[nueva_columna_intentos] = pd.to_numeric(df_modificado[nueva_columna_intentos], errors='coerce')
                
                nuevas_columnas_temp.append(nueva_columna_intentos)

            # Agregar solo las columnas únicas de esta ronda
            columnas_nuevas.extend(list(set(nuevas_columnas_temp)))

            print(f"[Incluido] {col_nota} y {col_intentos} → {proporcion:.2%} válidos → nuevas columnas creadas.")
        else:
            print(f"[Excluido] {col_nota} y {col_intentos} → {proporcion:.2%} válidos (<{umbral:.0%})")

    print("\nResumen final:")
    print(f"Columnas nuevas seleccionadas: {columnas_nuevas}")

    return columnas_nuevas, df_modificado



##La diferencia entre esta funcion y la atenrior es que esta tiene en cuenta que pueden venir diferentes asiganturas en las opciones de prerwequisito
import pandas as pd

def columnas_prereq_validas_ext_dif_prereq(
    df: pd.DataFrame,
    nombres_asignaturas: pd.DataFrame,
    umbral: float = 0.8,            # umbral sobre Prereq_i_Nota (columna fuente)
    umbral_pareja: float = 0.90     # umbral por par de columnas creadas (nota+intentos)
):
    """
    Identifica columnas de prerequisitos que cumplen con el umbral de no-nulos
    y crea nuevas columnas por cada CÓDIGO distinto, con el nombre de la asignatura
    (o el código si no hay match). Cada código genera un par:
        Prereq_<nombre_norm>_Nota
        Prereq_<nombre_norm>_Intentos

    Reglas adicionales:
      - Si para un código hay múltiples descripciones en 'nombres_asignaturas', se toma la primera.
      - Tras crear el par, si su completitud relativa es < umbral_pareja, se eliminan ambas columnas
        y no se reportan en la salida.

    Parámetros:
        df: DataFrame de entrada con columnas Prereq_i_Nota, Prereq_i_Codigo, Prereq_i_Intentos
        nombres_asignaturas: DataFrame con ["Cod materia curso", "Descripcion_Materia"]
        umbral: proporción mínima de valores no nulos en la columna fuente Prereq_i_Nota (default=0.8)
        umbral_pareja: proporción mínima (>=0.90) del par nuevo respecto a no-nulos de Prereq_i_Nota

    Retorna:
        columnas_nuevas (list): nombres de columnas finalmente creadas (y que superaron el umbral_pareja)
        df_modificado (pd.DataFrame): DataFrame con las columnas nuevas añadidas
    """
    columnas_nuevas = []
    df_modificado = df.copy()

    prereq_notas = [c for c in df.columns if c.startswith("Prereq_") and c.endswith("_Nota")]

    for col_nota in prereq_notas:
        col_intentos = col_nota.replace("_Nota", "_Intentos")
        col_codigo = col_nota.replace("_Nota", "_Codigo")

        if col_intentos not in df.columns:
            print(f"[Aviso] No se encontró la columna de intentos para {col_nota}, se omite el trío.")
            continue
        if col_codigo not in df.columns:
            print(f"[Aviso] No se encontró la columna de código para {col_nota}, se omite el trío.")
            continue

        total = len(df_modificado)
        no_nulos_fuente = df_modificado[col_nota].notna().sum()
        proporcion_fuente = (no_nulos_fuente / total) if total > 0 else 0.0

        if proporcion_fuente < umbral:
            print(f"[Excluido] {col_nota} (y su par {col_intentos}) → {proporcion_fuente:.2%} válidos (<{umbral:.0%})")
            continue

        # --- Paso 1: identificar códigos distintos en este prereq_i ---
        codigos_validos = df_modificado.loc[df_modificado[col_codigo].notna(), col_codigo].unique().tolist()

        # Guardar columnas creadas en esta ronda para posible limpieza por umbral_pareja
        columnas_creadas_ronda = []  # lista de pares (nota_col, intentos_col)

        # --- Paso 2: crear/llenar columnas por CÓDIGO ---
        for codigo in codigos_validos:
            # Buscar nombre (tomar primera coincidencia) o usar el propio código
            match = nombres_asignaturas.loc[
                nombres_asignaturas["Cod materia curso"] == codigo,
                "Descripcion_Materia"
            ]
            if not match.empty:
                nombre_materia = match.values[0]
            else:
                nombre_materia = codigo  # fallback

            # Normalizar nombre
            nombre_materia_norm = normalizar_nombre(str(nombre_materia))

            nueva_col_nota = f"Prereq_{nombre_materia_norm}_Nota"
            nueva_col_int = f"Prereq_{nombre_materia_norm}_Intentos"

            # Crear columnas si no existen
            if nueva_col_nota not in df_modificado.columns:
                df_modificado[nueva_col_nota] = pd.NA
            if nueva_col_int not in df_modificado.columns:
                df_modificado[nueva_col_int] = pd.NA

            # Filas donde este código aplica en este prereq
            mask_codigo = df_modificado[col_codigo] == codigo

            # Asignar valores sólo en las filas donde corresponde el código
            # Nota
            valores_nota = pd.to_numeric(df_modificado.loc[mask_codigo, col_nota], errors="coerce")
            df_modificado.loc[mask_codigo, nueva_col_nota] = valores_nota

            # Intentos
            valores_intentos = pd.to_numeric(df_modificado.loc[mask_codigo, col_intentos], errors="coerce")
            df_modificado.loc[mask_codigo, nueva_col_int] = valores_intentos

            # Forzar tipo numérico (permitiendo NaN)
            df_modificado[nueva_col_nota] = pd.to_numeric(df_modificado[nueva_col_nota], errors="coerce")
            df_modificado[nueva_col_int] = pd.to_numeric(df_modificado[nueva_col_int], errors="coerce")

            columnas_creadas_ronda.append((nueva_col_nota, nueva_col_int))

        # --- Paso 3: filtrar por completitud relativa (umbral_pareja) y limpiar pares que no cumplan ---
        # Denominador: no_nulos_fuente (Prerreq_i_Nota) – filas donde hubo dato en la fuente de este prereq
        if no_nulos_fuente == 0:
            # Nada que evaluar en esta ronda, pasar
            continue

        for nueva_col_nota, nueva_col_int in columnas_creadas_ronda:
            n_no_nulos_par = df_modificado[nueva_col_nota].notna().sum()
            completitud_relativa = n_no_nulos_par / no_nulos_fuente

            if completitud_relativa >= umbral_pareja:
                # Mantener y reportar
                columnas_nuevas.extend([nueva_col_nota, nueva_col_int])
                print(f"[Incluido] {nueva_col_nota} / {nueva_col_int} → "
                      f"{completitud_relativa:.2%} de {col_nota} (>= {umbral_pareja:.0%})")
            else:
                # Eliminar ambas columnas (par) y NO reportarlas
                df_modificado.drop(columns=[nueva_col_nota, nueva_col_int], inplace=True, errors="ignore")
                print(f"[Excluido] {nueva_col_nota} / {nueva_col_int} → "
                      f"{completitud_relativa:.2%} de {col_nota} (< {umbral_pareja:.0%})")

    # Dejar únicas y mantener orden de aparición
    seen = set()
    columnas_nuevas = [c for c in columnas_nuevas if not (c in seen or seen.add(c))]

    print("\nResumen final:")
    print(f"Columnas nuevas seleccionadas ({len(columnas_nuevas)}): {columnas_nuevas}")

    return columnas_nuevas, df_modificado




In [11]:
#Cambiar a category

def cambiar_a_category(df, cols):
    # Convierte columnas a category si existen

    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("category")
        else:
            print(f"Columna no encontrada: {c}")

    # Verificar tipos resultantes
    print(df[ [c for c in cols if c in df.columns] ].dtypes)  

    return df

In [25]:
#Guardar resultados dataframe - > csv

import os

def guardar_resultados(df,carpeta,nombre):
        """Guarda los resultados en un archivo Excel"""
        if df is None:
            print("No hay resultados para guardar")
            return
        

        # Crear carpeta si no existe
        os.makedirs(carpeta, exist_ok=True)

        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        #nombre_archivo = "Resultados v6/resultados_prerrequisitos_v6_cadenas_"+timestamp+".xlsx"
        nombre_archivo = carpeta+nombre+timestamp+".csv"

        try:
            # Limpiar columnas completamente vacías
            df = df.dropna(axis=1, how='all')
            print(f"✓ Guardando resultados en: {nombre_archivo} ...")
            #df.to_excel(nombre_archivo, index=False)
            df.to_csv(nombre_archivo, index=False, sep=';', float_format='%.3f', decimal=',')
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(df.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

In [27]:
# Obtener features del modelo guardado

import os
import json
import pandas as pd
import xgboost as xgb
from typing import List, Optional, Dict

def leer_features_modelo(path_modelo: str) -> List[str]:
    """
    Devuelve la lista de features que el modelo XGBoost espera.
    Intenta primero vía Booster.feature_names y, si no está,
    lee directamente del JSON.
    """
    # 1) Intento estándar con Booster
    booster = xgb.Booster()
    booster.load_model(path_modelo)
    if getattr(booster, "feature_names", None):
        return list(booster.feature_names)

    # 2) Fallback: leer el JSON directamente
    with open(path_modelo, "r", encoding="utf-8") as f:
        data = json.load(f)

    # En JSON nuevos suele venir en "learner" -> "feature_names" o en "feature_names"
    # Probamos ambas rutas.
    cand_keys = [
        ["learner", "feature_names"],
        ["feature_names"],
        ["Learner", "feature_names"],  # por si cambia capitalización en alguna versión
    ]
    for path in cand_keys:
        node = data
        ok = True
        for k in path:
            if isinstance(node, dict) and k in node:
                node = node[k]
            else:
                ok = False
                break
        if ok and isinstance(node, list):
            return node

    # Último recurso: puede que el modelo haya sido entrenado sin nombres y use f0..fN-1
    # En ese caso, reconstruimos con prefijo 'f'
    # Número de features puede obtenerse de attrs si existen; si no, inferimos por los splits.
    try:
        n_feat = int(data.get("learner", {}).get("gradient_booster", {})
                     .get("model", {}).get("num_feature", None))
    except Exception:
        n_feat = None

    if n_feat is not None:
        return [f"f{i}" for i in range(n_feat)]

    # Si no se logró, levantamos un error explícito.
    raise ValueError("No se pudieron recuperar los nombres de features del modelo.")

#print(leer_features_modelo("modelos_guardados_menos_periodos/MAT4258.json"))

In [28]:
# Helpers de rutas v2 y compatibilidad legacy
import os
import json
import time

VALID_MODELOS = {'catboost', 'xgboost', 'rf'}
VALID_VARIANTS = {'main', 'fallback'}
VALID_TASKS = {'prediccion_retiro', 'prediccion_nota'}

def get_model_dir(modelo, variant, task):
    if modelo not in VALID_MODELOS:
        raise ValueError(f'Modelo invalido: {modelo}')
    if variant not in VALID_VARIANTS:
        raise ValueError(f'Variant invalido: {variant}')
    if task not in VALID_TASKS:
        raise ValueError(f'Task invalido: {task}')
    return os.path.join('modelos_guardados_v2', modelo, variant, task)

def get_model_paths(asignatura, modelo, variant, task):
    base_dir = get_model_dir(modelo, variant, task)
    ext_map = {
        'xgboost': '.json',
        'rf': '.joblib',
        'catboost': '.cbm',
    }
    ext = ext_map.get(modelo, '')
    nombre = str(asignatura)
    model_path = os.path.join(base_dir, f'{nombre}{ext}')
    features_path = os.path.join(base_dir, f'{nombre}_features.json')
    meta_path = os.path.join(base_dir, f'{nombre}_meta.json')
    return {'model': model_path, 'features': features_path, 'meta': meta_path}

def _legacy_model_dir(modelo, task):
    if modelo == 'xgboost':
        return 'modelos_guardados_v2/prediccion_nota' if task == 'prediccion_nota' else 'modelos_guardados_v2/prediccion_retiro'
    if modelo == 'rf':
        return 'modelos_guardados_v2/prediccion_nota_rf' if task == 'prediccion_nota' else 'modelos_guardados_v2/prediccion_retiro_rf'
    if modelo == 'catboost':
        return 'modelos_guardados_v2/prediccion_nota_cat' if task == 'prediccion_nota' else 'modelos_guardados_v2/prediccion_retiro_cat'
    return None

def _find_legacy_model_file(asignatura, modelo, task):
    legacy_dir = _legacy_model_dir(modelo, task)
    if legacy_dir is None or not os.path.isdir(legacy_dir):
        return None
    if modelo == 'xgboost':
        exts = ('.json', '.model')
    elif modelo == 'rf':
        exts = ('.joblib',)
    else:
        exts = ('.cbm',)
    files = [
        f for f in os.listdir(legacy_dir)
        if str(asignatura) in f and f.endswith(exts)
    ]
    if not files:
        return None
    return os.path.join(legacy_dir, files[0])

def _infer_features_from_model(modelo, modelo_nombre):
    if modelo is None:
        return None
    if modelo_nombre == 'xgboost':
        try:
            booster = modelo.get_booster()
            feats = getattr(booster, 'feature_names', None)
            if feats:
                return list(feats)
        except Exception:
            return None
    if modelo_nombre == 'rf':
        feats = getattr(modelo, 'feature_names_in_', None)
        if feats is not None:
            return list(feats)
    if modelo_nombre == 'catboost':
        try:
            feats = modelo.get_feature_names()
            if feats:
                return list(feats)
        except Exception:
            return None
    return None

def guardar_features_meta(paths, features=None, meta=None):
    if features:
        with open(paths['features'], 'w', encoding='utf-8') as f:
            json.dump(list(features), f, ensure_ascii=False)
    if meta:
        with open(paths['meta'], 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False)

def cargar_features_modelo(asignatura, modelo, variant, task, modelo_cargado=None):
    paths = get_model_paths(asignatura, modelo, variant, task)
    if os.path.isfile(paths['features']):
        with open(paths['features'], 'r', encoding='utf-8') as f:
            return json.load(f)

    # fallback: leer del modelo guardado legacy o del modelo en memoria
    if modelo == 'xgboost':
        legacy_model_path = _find_legacy_model_file(asignatura, modelo, task)
        if legacy_model_path is not None:
            return leer_features_modelo(legacy_model_path)

    inferred = _infer_features_from_model(modelo_cargado, modelo)
    return inferred

In [33]:
# Alinear el df para poder ejecutar el modelo cargado

def alinear_dataframe_a_modelo(df: pd.DataFrame,
                               expected_features: list[str],
                               fill_value: float = 0.0) -> pd.DataFrame:
    """
    Crea un DataFrame alineado al modelo:
    - Agrega columnas faltantes con fill_value
    - Elimina columnas sobrantes
    - Reordena columnas según 'expected_features'
    OJO: Asegúrate de usar el mismo **pipeline** de preparación que en entrenamiento.
    """
    df2 = df.copy()

    # Añadir faltantes
    for col in expected_features:
        if col not in df2.columns:
            df2[col] = fill_value

    # Filtrar solo las esperadas y reordenar
    df2 = df2[expected_features]

    return df2

In [30]:
# CARGAR ARCHIVO
print("=== Reivsar asignaturas. Carga Archivo Maestro.   ===\n")

#Flag para usar parquet o csv. El parquet fue creado del CSV original importando desde el otro metodo que genera el documento.
parquet_use=True

# Solicitar archivo de historial
while True:
    try:
        if parquet_use:
            ruta_historial = hitorial_parqet
            df_historial = pd.read_parquet(ruta_historial)
        else:
            ruta_historial = historial
            df_historial = pd.read_csv(ruta_historial, sep=';')
        print(f"✓ Archivo de historial cargado: {len(df_historial)} registros")
        df_historial_asignaturas_nombres = crear_df_asignaturas(df_historial)
        break
    except Exception as e:
        print(f"Error al cargar historial: {e}")
        print("Intente nuevamente.\n")

=== Reivsar asignaturas. Carga Archivo Maestro.   ===

✓ Archivo de historial cargado: 517640 registros
👌 Creado DataFrame con las asignaturas unicas y sus nombres. Número de asignaturas únicas: 1401


In [61]:
#Limpiar y cargar DataFrame

cols_to_excl =[
        "Nombre_Programa",
        "_ Matricula detalle para analisis.Prof_Codigo",
        "_ Matricula detalle para analisis.Sexo",
        "_ Matricula detalle para analisis.Procedencia Categoria",
    ]

tipo_prereq="Prerrequisito cumplido"
#tipo_prereq="all"
ruta_modelos_guardados='modelos_guardados_v2\prediccion_nota'
#asig_usar_temp=elegir_asignaturas(df_historial,tipo_prereq,50)

#["FIS1023","MAT1111","FIS1033"]#
#asig_a_usar=elegir_asignaturas(df_historial,tipo_prereq,50) #["FIS1023","MAT1111"]#,"FIS1033"]#["FIS1023","MAT1111","EST7042","IST2089","MAT4011","IBA4032","MAT4258","MAT4260","FIS1033","FIS1043"]
asig_a_usar=elegir_asignaturas_rev_ya_creados(df_historial,tipo_prereq,ruta_modelos_guardados,5)


#df_usar = df_historial[
#    (df_historial["Observacion_Prerrequisito"] == "Prerrequisito cumplido") &
#    (df_historial["Cod materia curso"].isin(asig_a_usar))
#].copy()

df_usar = df_historial[
    ((df_historial["Observacion_Prerrequisito"] == "Prerrequisito cumplido") | (df_historial["Observacion_Prerrequisito"] == "No tiene pre requisito")
      ) &
    (df_historial["Cod materia curso"].isin(asig_a_usar) )
].copy()

df_usar=arreglar_comas_por_puntos(df_usar,cols_to_excl)
df_usar=limpiar_dataframe(df_usar, True)

Lista de las asignaturas en la carpeta 


,asignatura
0,ADM0016
1,ADM0017
2,ADM0018
3,ADM0022
4,ADM0033
...,...
402,SCO1010
403,SCO4025
404,SCO4090
405,TCH4000


Lista de asignaturas seleccionadas: 
     Cod materia curso  num_registros
202           ECO8731            274
203           ADM0061            273
204           TCH4002            272
205           FRA7021            271
206           ODO4010            270
[Comas→Puntos] Columna excluida: Nombre_Programa
[Comas→Puntos] Procesada columna: Prereq_1_Codigo
[Comas→Puntos] Procesada columna: Prereq_1_Tipo
[Comas→Puntos] Procesada columna: Prereq_1_EsCadena
[Comas→Puntos] Procesada columna: Prereq_2_Codigo
[Comas→Puntos] Procesada columna: Prereq_2_Tipo
[Comas→Puntos] Procesada columna: Prereq_2_EsCadena
[Comas→Puntos] Procesada columna: Prereq_3_Codigo
[Comas→Puntos] Procesada columna: Prereq_3_Tipo
[Comas→Puntos] Procesada columna: Prereq_3_EsCadena
[Comas→Puntos] Procesada columna: Prereq_4_Codigo
[Comas→Puntos] Procesada columna: Prereq_4_Tipo
[Comas→Puntos] Procesada columna: Prereq_4_EsCadena
[Comas→Puntos] Procesada columna: Prereq_5_Codigo
[Comas→Puntos] Procesada columna: Prereq_

In [ ]:

# Helpers para usar modelos guardados CatBoost
import os
import numpy as np
import pandas as pd
import shap
from catboost import CatBoostRegressor, CatBoostClassifier, Pool

def cargar_modelo_catboost_regresion(ruta_carpeta, codigo_asignatura, variant='main'):
    paths = get_model_paths(codigo_asignatura, 'catboost', variant, 'prediccion_nota')
    ruta_modelo = paths['model']
    if not os.path.isfile(ruta_modelo):
        legacy_path = _find_legacy_model_file(codigo_asignatura, 'catboost', 'prediccion_nota')
        if legacy_path is not None:
            ruta_modelo = legacy_path
        else:
            print(f'[Aviso] No se encontro modelo CatBoost de regresion para {codigo_asignatura}')
            return None

    modelo = CatBoostRegressor()
    modelo.load_model(ruta_modelo)
    print(f'[Info] Modelo de regresion CatBoost cargado desde {ruta_modelo}')
    return modelo


def cargar_modelo_catboost_clasificacion(ruta_carpeta, codigo_asignatura, variant='main'):
    paths = get_model_paths(codigo_asignatura, 'catboost', variant, 'prediccion_retiro')
    ruta_modelo = paths['model']
    if not os.path.isfile(ruta_modelo):
        legacy_path = _find_legacy_model_file(codigo_asignatura, 'catboost', 'prediccion_retiro')
        if legacy_path is not None:
            ruta_modelo = legacy_path
        else:
            print(f'[Aviso] No se encontro modelo CatBoost de clasificacion para {codigo_asignatura}')
            return None

    modelo = CatBoostClassifier()
    modelo.load_model(ruta_modelo)
    print(f'[Info] Modelo de clasificacion CatBoost cargado desde {ruta_modelo}')
    return modelo


def obtener_features_modelo_catboost(modelo, fallback_cols):
    try:
        features = modelo.get_feature_names()
        if features:
            return list(features)
    except Exception:
        pass
    return list(fallback_cols)


def escribir_interpretaciones_shap_catboost_modelo_cargado(modelo, X, df, top_n=10, col_general='interpretacion_general', col_registro='interpretacion_registro'):
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    try:
        cat_idx = list(modelo.get_cat_feature_indices())
    except Exception:
        cat_idx = []

    if cat_idx:
        pool = Pool(X, cat_features=cat_idx)
        explainer = shap.TreeExplainer(modelo)
        shap_exp = explainer(pool)
        vals = shap_exp.values
    else:
        explainer = shap.TreeExplainer(modelo)
        shap_exp = explainer(X)
        vals = shap_exp.values

    importancia_media = pd.Series(np.abs(vals).mean(axis=0), index=X.columns).sort_values(ascending=False)
    top_imp = importancia_media.head(top_n)
    interp_general = 'Variables con mayor impacto promedio (|SHAP|): ' + ', '.join(
        f'{feat} ({imp:.3f})' for feat, imp in top_imp.items()
    )
    general_series = pd.Series(interp_general, index=X.index)
    print('[Info] Interpretacion general generada.')
    print(interp_general)

    def resumen_por_fila(row_vals):
        contrib = pd.Series(row_vals, index=X.columns)
        pos = contrib[contrib > 0].abs().sort_values(ascending=False).head(top_n)
        neg = contrib[contrib < 0].abs().sort_values(ascending=False).head(top_n)
        pos_str = ', '.join(f'{f} (+{contrib[f]:.3f})' for f in pos.index) if len(pos) else '0'
        neg_str = ', '.join(f'{f} ({contrib[f]:.3f})' for f in neg.index) if len(neg) else '0'
        return f'A favor: {pos_str} | En contra: {neg_str}'

    print('[Info] Generando resumen por fila...')
    interp_registro_series = pd.Series(
        (resumen_por_fila(vals[i, :]) for i in range(vals.shape[0])),
        index=X.index
    )

    if len(df) == len(X) and df.index.equals(X.index):
        df[col_general] = general_series
        df[col_registro] = interp_registro_series
    else:
        try:
            df.loc[X.index, col_general] = general_series
            df.loc[X.index, col_registro] = interp_registro_series
        except Exception:
            if len(df) == len(X):
                df[col_general] = general_series.values
                df[col_registro] = interp_registro_series.values
            else:
                raise ValueError('No coinciden longitudes/indices de df y X; no se pueden escribir interpretaciones de forma segura.')

    return df, importancia_media


In [ ]:

## Usar modelos guardados catboost
from catboost import Pool

def usar_modelos_guardados_catboost_por_asignatura(df_usar):
    col_usar = []
    var_objetivo = ''
    df_resultados_final = pd.DataFrame()
    df_errores = pd.DataFrame()
    asig_a_usar = df_usar['Cod materia curso'].unique().tolist()

    ruta_modelos_regresion = get_model_dir('catboost', 'main', 'prediccion_nota')
    ruta_modelos_clasificacion = get_model_dir('catboost', 'main', 'prediccion_retiro')

    for asig in asig_a_usar:
        try:
            nombre_asig = df_usar[df_usar['Cod materia curso'] == asig]['Descripcion_Materia'].iloc[0]
            print(f'\n == Resultados para programa:  {asig} - {nombre_asig} == \n')
            df_usar_filtrado = df_usar[df_usar['Cod materia curso'] == asig].copy()
            tipo_asig_prereq = df_usar_filtrado['Observacion_Prerrequisito'].iloc[0]
            col_usar = []
            var_objetivo = ''
            lista_prereq_usar = []
            var_objetivo_clas = 'Retiro_Asignatura_Cat'

            if tipo_asig_prereq == 'Prerrequisito cumplido':
                df_usar_filtrado, col_usar, var_objetivo = renombrar_columnas(df_usar_filtrado, tiene_prereq=True)
                lista_prereq_usar, df_usar_filtrado = columnas_prereq_validas_ext_dif_prereq(
                    df_usar_filtrado, df_historial_asignaturas_nombres, 0.8
                )
                col_usar = col_usar + lista_prereq_usar
                print(f'Columnas a usar ({len(col_usar)}): {col_usar} \n  Numero de filas a tener en cuenta: {len(df_usar_filtrado)}')
            else:
                df_usar_filtrado, col_usar, var_objetivo = renombrar_columnas(df_usar_filtrado, tiene_prereq=False)
                lista_prereq_usar = columnas_prereq_validas(df_usar_filtrado, 0.8)
                col_usar = col_usar + lista_prereq_usar
                print(f'Columnas a usar ({len(col_usar)}): {col_usar} \n  Numero de filas a tener en cuenta: {len(df_usar_filtrado)}')

            cols_to_category = [
                'programa',
                'sexo',
                'procedencia_categoria',
                'profesor_codigo',
                'Tipo_colegio',
                'Tipo_calendario'
            ]

            cols_present = [c for c in col_usar if c in df_usar_filtrado.columns]
            if var_objetivo in df_usar_filtrado.columns and var_objetivo not in cols_present:
                cols_present.append(var_objetivo)

            faltantes = [c for c in col_usar if c not in df_usar_filtrado.columns]
            if faltantes:
                print(f'[Aviso] Algunas columnas de col_usar no existen y se omiten: {faltantes}')

            df_usar_filtrado = df_usar_filtrado[cols_present].copy()
            print(f'[Info] df_usar_filtrado reducido a {df_usar_filtrado.shape[1]} columnas y {len(df_usar_filtrado)} filas')

            df_usar_filtrado, num_filas_eliminadas = eliminar_filas_por_columna(df_usar_filtrado)
            df_usar_filtrado = cambiar_a_category(df_usar_filtrado, cols_to_category)

            # Cargar modelos CatBoost
            modelo_reg = cargar_modelo_catboost_regresion(ruta_modelos_regresion, asig)
            if modelo_reg is None:
                print('[Aviso] Esta asignatura no tiene modelo de regresion CatBoost.')
                continue

            features_modelo = cargar_features_modelo(asig, 'catboost', 'main', 'prediccion_nota', modelo_cargado=modelo_reg)
            print(f'Las columnas esperadas (regresion) son {features_modelo}')

            col_variable_obj = df_usar_filtrado[var_objetivo] if var_objetivo in df_usar_filtrado.columns else None
            col_variable_obj_clas = df_usar_filtrado[var_objetivo_clas] if var_objetivo_clas in df_usar_filtrado.columns else None

            df_usar_filtrado = alinear_dataframe_a_modelo(df_usar_filtrado, features_modelo, fill_value=0.0)

            if col_variable_obj is not None:
                df_usar_filtrado[var_objetivo] = col_variable_obj
            if col_variable_obj_clas is not None:
                df_usar_filtrado[var_objetivo_clas] = col_variable_obj_clas

            print(f'Las columnas en df_usar_filtrado tras alinear son {df_usar_filtrado.columns}')

            # Clasificacion de retiros
            modelo_clasif = cargar_modelo_catboost_clasificacion(ruta_modelos_clasificacion, asig)
            if modelo_clasif is None:
                print('[Aviso] Esta asignatura no tiene modelo de clasificacion CatBoost.')
                col_prediccion_clasif = 'Clasificacion_CAT'
                df_clasif = df_usar_filtrado.copy()
                df_clasif[col_prediccion_clasif] = 0
            else:
                X_clasif = df_usar_filtrado[features_modelo]
                try:
                    cat_idx = list(modelo_clasif.get_cat_feature_indices())
                except Exception:
                    cat_idx = []
                if cat_idx:
                    pool_clasif = Pool(X_clasif, cat_features=cat_idx)
                    y_pred = modelo_clasif.predict(pool_clasif)
                else:
                    y_pred = modelo_clasif.predict(X_clasif)
                y_pred = np.array(y_pred).astype(int).ravel()
                df_clasif = df_usar_filtrado.copy()
                col_prediccion_clasif = 'Clasificacion_CAT'
                df_clasif[col_prediccion_clasif] = y_pred

            # Regresion de nota final
            X_reg = df_clasif[features_modelo]
            try:
                cat_idx_reg = list(modelo_reg.get_cat_feature_indices())
            except Exception:
                cat_idx_reg = []
            if cat_idx_reg:
                pool_reg = Pool(X_reg, cat_features=cat_idx_reg)
                y_pred_reg = modelo_reg.predict(pool_reg)
            else:
                y_pred_reg = modelo_reg.predict(X_reg)

            df_pred = df_clasif.copy()
            col_prediccion = 'Prediccion_CAT'
            df_pred[col_prediccion] = y_pred_reg

            # Interpretaciones SHAP (solo regresion)
            cols_to_category = cols_to_category + ['interpretacion_general', 'interpretacion_registro']
            df_pred = cambiar_a_category(df_pred, cols_to_category)
            print('[Aviso] Insertando interpretaciones del modelo de regresion en el dataframe:')
            df_pred, importanciamedia = escribir_interpretaciones_shap_catboost_modelo_cargado(
                modelo_reg, X_reg, df_pred, top_n=10, col_general='interpretacion_general', col_registro='interpretacion_registro'
            )

            df_pred['Descripcion_Materia'] = nombre_asig
            cols_to_add = ['Periodo', 'Nombre_Division', 'cohorte', 'Pidm', 'nrc', 'DPTO Asignatura']
            col_pred_final = 'Prediccion_final'

            print('[Aviso] Insertando columna de conclusion del modelo en el dataframe')
            if col_prediccion in df_pred.columns:
                df_pred[col_prediccion] = df_pred[col_prediccion].round(2)
                df_pred[col_prediccion] = df_pred[col_prediccion].apply(lambda x: -1 if x < 0 else x)
                df_pred[col_prediccion] = df_pred[col_prediccion].apply(lambda x: 5 if x > 5 else x)
                df_pred[col_pred_final] = df_pred.apply(
                    lambda row: -1 if row[col_prediccion_clasif] > 0 else row[col_prediccion], axis=1
                )

            df_to_add = df_usar[cols_to_add].reindex(df_pred.index)
            df_pred = df_pred.join(df_to_add)

            if not isinstance(df_resultados_final, pd.DataFrame):
                try:
                    df_resultados_final = pd.DataFrame(df_resultados_final)
                except Exception:
                    df_resultados_final = pd.DataFrame()

            df_resultados_final = pd.concat([df_resultados_final, df_pred], ignore_index=True, sort=False)
            print('[Aviso] df_resultados_final concatenado')
        except Exception as e:
            print(f'Error al procesar la asignatura {asig}: {e}')
            if df_errores.empty:
                df_errores = pd.DataFrame()
            nombre_asig = df_usar[df_usar['Cod materia curso'] == asig]['Descripcion_Materia'].iloc[0]
            nombre_asig = pd.DataFrame([{'Cod materia curso': asig, 'Descripcion_Materia': nombre_asig, 'Error': f'{type(e).__name__}: {e}'}])
            df_errores = pd.concat([df_errores, nombre_asig], ignore_index=True, sort=False)

    return df_resultados_final, df_errores


In [ ]:
# Ejecutar modelos guardados y exportar resultados
df_resultados_final, df_errores = usar_modelos_guardados_catboost_por_asignatura(df_usar)

# Exportar resultados
carpeta_salida = 'Resultados_Modelo_Excel_MegaPipeline/'
guardar_resultados(df_resultados_final, carpeta_salida, 'catboost_resultados_')
guardar_resultados(df_errores, carpeta_salida, 'catboost_errores_')
